<a href="https://colab.research.google.com/github/Matrix-69/GenAI/blob/main/GenAI_GitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade --force-reinstall langchain langchain-community langchain-openai faiss-cpu pdfplumber requests

In [ ]:
  !pip install gensim


In [ ]:
    !pip install wikipedia-api


    Now that the necessary libraries are installed, I'll modify the cell `V7woHB24iyVS` to include the `gensim.downloader` import, ensuring `api` is defined.
    

In [ ]:
    %history -g V7woHB24iyVS
    # The previous cell output was not captured correctly, so I'm using %history to retrieve the content of cell V7woHB24iyVS to prepare for modification.
    # I will then use modify_cells to fix it in the next step.


In [ ]:
import gensim.downloader as api
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

model = api.load('word2vec-google-news-300')

words = ['king', 'man', 'woman', 'queen']
vectors = [model[word] for word in words]

king = model['king']
man = model['man']
woman = model['woman']

result_vector = king - man + woman

similar_words = model.most_similar(result_vector, topn=5)

print("Most similar words:")
for word, sim in similar_words:
    print(word, sim)

pca = PCA(n_components=2)
result = pca.fit_transform(vectors)

plt.scatter(result[:, 0], result[:, 1])
for i, word in enumerate(words):
    plt.text(result[i, 0], result[i, 1], word)

plt.show()

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

sports_words = ['soccer', 'football', 'basketball', 'player', 'team',
                'coach', 'referee', 'goal', 'championship', 'league']

vectors = [model[word] for word in sports_words]

pca = PCA(n_components=2)
pca_result = pca.fit_transform(vectors)

plt.scatter(pca_result[:, 0], pca_result[:, 1])

for i, word in enumerate(sports_words):
    plt.text(pca_result[i, 0], pca_result[i, 1], word)

plt.show()

def find_similar(word):
    print(model.most_similar(word, topn=5))

find_similar('soccer')

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import string

corpus = [
    "The plaintiff filed a lawsuit against the defendant",
    "The court will hear the case",
    "The lawyer presented evidence",
    "The judge ruled in favor of plaintiff"
]

def preprocess(text):
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text.lower())
    return [w for w in tokens if w not in stop_words and w not in string.punctuation]

processed = [preprocess(doc) for doc in corpus]

model2 = Word2Vec(processed, vector_size=100, window=5, min_count=1)

print(model2.wv.most_similar('plaintiff'))

In [ ]:
import random
import gensim.downloader as api
word_vectors = api.load("glove-wiki-gigaword-50")

def get_similar(word):
    try:
        return [w[0] for w in word_vectors.most_similar(word, topn=3)]
    except:
        return []

def enrich(prompt):
    words = prompt.split()
    new_words = []
    for w in words:
        sim = get_similar(w)
        new_words.append(random.choice(sim) if sim else w)
    return " ".join(new_words)

prompt = "Write a short story about a brave warrior fighting a dragon"

print("Original:", prompt)
print("Enriched:", enrich(prompt))

In [ ]:
def create_story(seed):
    sim = get_similar(seed)
    if len(sim) < 5:
        return "Try another word"

    return f"In a world of {sim[0]} and {sim[1]}, {seed} led a journey full of {sim[2]} and {sim[3]} facing {sim[4]} challenges."

print(create_story("adventure"))

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

texts = [
    "Amazing product!",
    "Worst service ever",
    "It was okay"
]

for t in texts:
    print(t, sentiment(t)[0])

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis")

texts = [
    "Amazing product!",
    "Worst service ever",
    "It was okay"
]

for t in texts:
    print(t, sentiment(t)[0])

In [ ]:
import wikipediaapi
from pydantic import BaseModel

class Institution(BaseModel):
    name: str
    summary: str

wiki = wikipediaapi.Wikipedia(user_agent='Colab_Wikipedia_API_Example/1.0 (https://colab.research.google.com)', language='en')

name = "MIT"
page = wiki.page(name)

data = Institution(name=name, summary=page.summary[:500])

print(data.model_dump_json(indent=4))

In [ ]:
!pip install langchain-text-splitters

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
!pip install -U langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pdfplumber requests

In [ ]:
# =========================
# IPC CHATBOT - FINAL (FIXED PDF ERROR)
# =========================

import os
import requests
import pdfplumber

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 🔑 API KEY
os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# -------------------------
# STEP 1: DOWNLOAD PDF (FIXED)
# -------------------------
IPC_URL = "https://www.mha.gov.in/sites/default/files/IPAct_1860.pdf"
IPC_FILE = "ipc.pdf"

def download_pdf(url, filename):
    print("Downloading IPC document...")

    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(url, headers=headers)

    # ✅ Check if actually PDF
    if "application/pdf" not in response.headers.get("Content-Type", ""):
        raise ValueError("Downloaded file is not a PDF. URL may be blocked or redirected.")

    with open(filename, "wb") as f:
        f.write(response.content)

    print("Download successful")

if not os.path.exists(IPC_FILE):
    download_pdf(IPC_URL, IPC_FILE)

# -------------------------
# STEP 2: EXTRACT TEXT
# -------------------------
print("Extracting text...")

print("Text length:", len(ipc_text))

# -------------------------
# STEP 3: SPLIT TEXT
# -------------------------
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

chunks = splitter.split_text(ipc_text)

# Optional optimization
# chunks = chunks[:200]

print("Chunks:", len(chunks))

# -------------------------
# STEP 4: VECTOR STORE
# -------------------------
print("Creating embeddings...")

# -------------------------
# STEP 5: QA SYSTEM
# -------------------------
llm = ChatOpenAI(model="gpt-3.5-turbo")


print("\n✅ Chatbot Ready (type 'exit')\n")

# -------------------------
# STEP 6: CHAT LOOP
# -------------------------
while True:
    query = input("Ask: ")

    if query.lower() == "exit":
        break

    try:
        print("Answer:", qa.run(query), "\n")
    except Exception as e:
        print("Error:", e)